# 02 · Entrenamiento

## ¿Qué hace el entrenamiento?
Mostrarle a la red miles de imágenes con su respuesta correcta y ajustar sus pesos para que se equivoque menos.
Cada vuelta completa sobre el entrenamiento es una **época**. Tras cada época se mide el error en **validación**
(imágenes que la red no usa para aprender) para decidir cuándo parar y qué versión guardar.

| Conjunto | Para qué sirve | Quién lo usa |
|---|---|---|
| entrenamiento | la red **aprende** con esto | el entrenamiento |
| validación | **elegir** (cuándo parar, qué época guardar, qué lr) | tú, durante el desarrollo |
| prueba | **informar** el resultado final, UNA sola vez | solo al terminar |

## Las dos fases (transfer learning)
1. **Fase 1:** la base MobileNetV2 (ya entrenada con ImageNet) está *congelada*; solo aprende la cabeza nueva (lr 1e-3).
2. **Fase 2 (ajuste fino):** se liberan las últimas 30 capas de la base con lr muy bajo (1e-5) para adaptarlas.

## Normalización
La red espera valores en un rango concreto. MobileNetV2 usa `[-1, 1]`: `x/127.5 − 1`. Está **dentro del modelo**, justo
*después* de la aumentación (que trabaja en 0–255). Si fuera al revés, `RandomBrightness` quedaría roto.

In [ ]:
# ===================== CELDA DE ARRANQUE: no hay que editar nada =====================
# Monta tu Drive, encuentra sola la carpeta del proyecto y Surface Crack, y deja todo listo.
from google.colab import drive
drive.mount("/content/drive")

import glob, os, sys
_raiz = "/content/drive/MyDrive"
_cands = sorted(glob.glob(f"{_raiz}/*/src/datos.py") + glob.glob(f"{_raiz}/*/*/src/datos.py"),
                key=os.path.getmtime, reverse=True)
if not _cands:
    raise FileNotFoundError("No encontré la carpeta del proyecto NUEVO en tu Drive (busco una carpeta que tenga src/datos.py). "
                            "Subí la carpeta del zip nuevo a tu Drive y volvé a ejecutar esta celda.")
PROYECTO = os.path.dirname(os.path.dirname(_cands[0]))
os.chdir(PROYECTO)
sys.path.insert(0, PROYECTO)
print("Proyecto:", PROYECTO)
!pip -q install scikit-image scikit-learn pandas matplotlib

from src import colab
RUTA_DATOS = colab.preparar(PROYECTO, _raiz)       # Surface Crack (zip -> disco local, o carpeta en Drive)

import json, shutil, zipfile
from pathlib import Path
import numpy as np, pandas as pd
from src import config, datos, evaluar, entrenar, modelo

import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## Paso 1 · Cargar las particiones del EDA
Mismas particiones que se guardaron en el cuaderno 01. Nada se vuelve a dividir aquí.

In [ ]:
tr = pd.read_csv("results/particion_entrenamiento.csv")
va = pd.read_csv("results/particion_validacion.csv")
datos.verificar_sin_fuga(tr, va)
print(datos.resumen_particiones(entrenamiento=tr, validacion=va))

## Paso 2 · (Corrección del falso positivo) Sobremuestrear los negativos difíciles
**Qué hace:** repite varias veces las filas de la fuente `negativos_dificiles` en el entrenamiento. Sin esto, 300
imágenes entre 40.000 casi no influyen.

**Cuidado:** si se repiten demasiado, la red las **memoriza**. Por eso la validación debe tener *escenas distintas*
(se logra con `grupos: "subcarpeta"`). Si no tenés aún esa fuente, saltá la celda.

In [ ]:
if "negativos_dificiles" in set(tr["fuente"]):
    tr = entrenar.sobremuestrear(tr, "negativos_dificiles", veces=5)
    print("Entrenamiento tras sobremuestrear:", len(tr))
else:
    print("Todavía no hay negativos difíciles: se entrena solo con Surface Crack.")

## Paso 3 · Barrido de la tasa de aprendizaje (rápido)
**Qué hace:** entrena 5 épocas con 8.000 imágenes para cada lr y mira el `val_loss`. La tasa de aprendizaje es
*cuánto se mueven los pesos en cada paso*: muy alta, la red se descontrola; muy baja, aprende lentísimo.

**Qué decidir:** usar el lr de menor `val_loss_min` como `lr1`. Es una prueba para **elegir**, no el entrenamiento final.

In [ ]:
barrido = entrenar.barrido_lr(tr, va, lrs=(1e-2, 1e-3, 1e-4))
display(barrido)
LR1 = float(barrido.loc[0, "lr"])
print("lr elegido para la fase 1:", LR1)

## Paso 4 · Entrenar MobileNetV2
**Qué hace:** fase 1 + fase 2, guardando el mejor modelo por `val_loss`, **un historial por fase** (`append=False`: se
acabó el historial con filas de un intento interrumpido) y un `meta.json` con todo lo necesario para reproducir.

**Paciencia:** `paciencia=50` en la corrida anterior significó, en la práctica, *no parar antes*. La parada temprana con
paciencia 2 cortaba durante una meseta temporal. Se deja igual para que sea comparable.

In [ ]:
meta = entrenar.entrenar("v2_mobilenet", tr, va, arquitectura="mobilenetv2", lr1=LR1,
                         epocas1=60, epocas2=40, paciencia=50)
print(meta)

## Paso 5 · Entrenar el segundo modelo (EfficientNetB0)
**Por qué:** el profesor advirtió que si todos usan el mismo modelo la nota se reparte. Comparar **MobileNetV2 vs
EfficientNetB0** con la misma división, semilla y aumentación es un experimento limpio (cambia **un** factor).
EfficientNetB0 es más precisa pero más pesada: la comparación debe incluir **tamaño y tiempo de inferencia**, no solo exactitud.

In [ ]:
meta2 = entrenar.entrenar("v2_efficientnet", tr, va, arquitectura="efficientnetb0", lr1=LR1,
                          epocas1=60, epocas2=40, paciencia=50)

## Paso 6 · Curvas de entrenamiento
**Qué mirar:** `val_loss` bajando y luego estable. Si `loss` baja pero `val_loss` sube → sobreajuste.
**Recordá:** estas curvas **no detectan** problemas de dominio (la validación sale de las mismas fotos). Eso lo
hace el cuaderno 03.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for nombre in ("v2_mobilenet", "v2_efficientnet"):
    for fase in (1, 2):
        ruta = Path("models") / nombre / f"historial_fase{fase}.csv"
        if ruta.exists():
            h = pd.read_csv(ruta)
            ax[0].plot(h["val_loss"], label=f"{nombre} f{fase}")
            ax[1].plot(h["val_recall"], label=f"{nombre} f{fase}")
ax[0].set_title("val_loss"); ax[1].set_title("val_recall")
for a in ax: a.legend()
fig.tight_layout(); fig.savefig("results/curvas.png", dpi=150)

## Paso 7 · Evaluación en validación con intervalo de confianza
**Qué hace:** matriz de confusión y métricas. El **recall** (sensibilidad) importa más que la exactitud: un falso
negativo es una grieta que no se vio. Con el intervalo de Wilson se reporta la **incertidumbre**: "2 errores de
3.000" no es "0,07 % exacto".

La **prueba** (test) se evalúa en el cuaderno 03, una sola vez.

In [ ]:
from tensorflow import keras
for nombre, arq in [(n, a) for n, a in (("v2_mobilenet", "mobilenetv2"), ("v2_efficientnet", "efficientnetb0")) if Path(f"models/{n}/modelo.keras").exists()]:
    m = keras.models.load_model(f"models/{nombre}/modelo.keras")
    inf = modelo.modelo_inferencia(m, arq)
    prob = entrenar.predecir_rutas(entrenar.predictor(inf), va["ruta"].tolist())
    r = evaluar.metricas(va["etiqueta"].to_numpy(), prob)
    lo, hi = evaluar.intervalo_wilson(r["fn"], r["fn"] + r["tp"])
    print(f"{nombre}: exactitud {r['exactitud']:.4f} | recall {r['recall']:.4f} | falsos neg. {r['fn']} "
          f"(tasa {100*r['fn']/(r['fn']+r['tp']):.2f} %, IC95 {100*lo:.2f}–{100*hi:.2f} %) | falsos pos. {r['fp']}")

## Paso 8 · Exportar para el celular (TFLite, float16)
**Qué hace:** convierte el modelo (sin aumentación ni dropout) a TFLite con pesos en float16 (mitad de tamaño).

In [ ]:
m = keras.models.load_model("models/v2_mobilenet/modelo.keras")
tam = modelo.exportar_tflite(m, "models/v2_mobilenet/modelo.tflite", "mobilenetv2")
print(f"modelo.tflite: {tam/1e6:.2f} MB")